<a href="https://colab.research.google.com/github/kaylatucker/CS690_Capstone/blob/main/03_notebooks/09_methods_and_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Candidate Statistical Methods

Based on the exploratory analysis, the project will use statistical methods that match the structure and distributions of the data rather than selecting methods only for variety.

## Customer Experience

The primary outcome is `review_score`.

Potential methods:
- Group comparisons for review scores across delivery status and other categorical variables.
- Spearman correlation for relationships involving review score and continuous variables because several variables are highly skewed and review score is ordinal.
- Nonparametric comparisons such as the Mann-Whitney U test when comparing review-score distributions between two groups.
- A multivariable method may later be used to examine delivery, order value, freight, product category, seller characteristics, and geography together.

## Seller Performance

Seller performance is represented primarily by sales and order activity.

Potential methods:
- Spearman correlation for relationships between seller performance and continuous seller characteristics because seller measures are strongly right-skewed.
- Group comparisons where categorical variables such as geography are relevant.
- Later multivariable analysis may account for seller scale, product variety, freight, reviews, delivery performance, and geography simultaneously.

The initial baseline will remain simple. More complex methods will only be added if they provide information beyond the baseline.

# Code

In [1]:
!git clone https://github.com/kaylatucker/CS690_Capstone.git

Cloning into 'CS690_Capstone'...
remote: Enumerating objects: 217, done.
remote: Counting objects: 100% (177/177), done.
remote: Compressing objects: 100% (165/165), done.
remote: Total 217 (delta 94), reused 23 (delta 11), pack-reused 40 (from 4)
Receiving objects: 100% (217/217), 85.23 MiB | 13.41 MiB/s, done.
Resolving deltas: 100% (96/96), done.
Updating files: 100% (47/47), done.


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

REPO_DIR = Path("/content/CS690_Capstone")
PROCESSED_DIR = REPO_DIR / "02_data" / "processed"

customer_experience = pd.read_csv(
    PROCESSED_DIR / "customer_experience.csv"
)

seller_metrics = pd.read_csv(
    PROCESSED_DIR / "seller_metrics.csv"
)

print("Customer-experience rows:", len(customer_experience))
print("Seller rows:", len(seller_metrics))

Customer-experience rows: 99441
Seller rows: 3095


### Customer-experience baseline

In [3]:
baseline_customer = customer_experience.dropna(
    subset=["review_score", "late_delivery_flag"]
).copy()

print("Baseline orders:", len(baseline_customer))

Baseline orders: 95830


In [4]:
baseline_delivery = (
    baseline_customer
    .groupby("late_delivery_flag")["review_score"]
    .agg(["count", "mean", "median", "std"])
    .round(2)
)

baseline_delivery.index = ["On Time", "Late"]

display(baseline_delivery)

,count,mean,median,std
On Time,88168,4.29,5.0,1.15
Late,7662,2.57,2.0,1.66


In [5]:
on_time_mean = baseline_delivery.loc["On Time", "mean"]
late_mean = baseline_delivery.loc["Late", "mean"]

mean_difference = round(on_time_mean - late_mean, 2)

print("Difference in average review score:", mean_difference)

Difference in average review score: 1.72


In [6]:
from scipy.stats import mannwhitneyu

on_time_reviews = baseline_customer.loc[
    baseline_customer["late_delivery_flag"] == 0,
    "review_score"
]

late_reviews = baseline_customer.loc[
    baseline_customer["late_delivery_flag"] == 1,
    "review_score"
]

u_stat, p_value = mannwhitneyu(
    on_time_reviews,
    late_reviews,
    alternative="two-sided"
)

print("Mann-Whitney U statistic:", u_stat)
print("p-value:", p_value)

Mann-Whitney U statistic: 524826310.5
p-value: 0.0


In [7]:
from scipy.stats import spearmanr

delay_sample = customer_experience[
    ["review_score", "delivery_delay_days"]
].dropna()

rho, p_value = spearmanr(
    delay_sample["delivery_delay_days"],
    delay_sample["review_score"]
)

print("Spearman correlation:", round(rho, 3))
print("p-value:", p_value)
print("Orders used:", len(delay_sample))

Spearman correlation: -0.176
p-value: 0.0
Orders used: 95830



The initial customer-experience baseline compares review scores for on-time and late orders. On-time orders have an average review score of approximately 4.29, compared with approximately 2.57 for late orders, a difference of about 1.72 review points.

A nonparametric group comparison is used because review score is bounded and ordinal rather than normally distributed. Delivery delay is also examined using Spearman correlation because it does not require normally distributed variables.

This baseline is intentionally simple. It establishes whether delivery performance has a meaningful relationship with customer satisfaction before later analyses account for order value, freight, product category, seller characteristics, and geography.

### Seller-performance baseline

In [9]:
from scipy.stats import spearmanr

seller_baseline_fields = [
    "seller_total_sales",
    "seller_order_count",
    "seller_product_count",
    "seller_category_count",
    "seller_avg_item_value"
]

seller_baseline = seller_metrics[
    seller_baseline_fields
].dropna()

print("Sellers used:", len(seller_baseline))

Sellers used: 3095


In [10]:
seller_results = []

for variable in [
    "seller_order_count",
    "seller_product_count",
    "seller_category_count",
    "seller_avg_item_value"
]:
    rho, p_value = spearmanr(
        seller_baseline[variable],
        seller_baseline["seller_total_sales"]
    )

    seller_results.append({
        "variable": variable,
        "spearman_rho": round(rho, 3),
        "p_value": p_value
    })

seller_baseline_results = pd.DataFrame(seller_results)

display(seller_baseline_results)

,variable,spearman_rho,p_value
0,seller_order_count,0.853,0.000000e+00
1,seller_product_count,0.786,0.000000e+00
2,seller_category_count,0.506,4.249797e-201
3,seller_avg_item_value,0.490,2.419164e-186


# Notes

- The methods and baseline analysis used the final customer-experience and seller-performance datasets.
- Statistical methods were selected based on the structure and distributions observed during EDA rather than for the purpose of using a specific technique.
- Nonparametric methods were emphasized because review scores are ordinal and several customer and seller variables are strongly skewed.

## Customer-Experience Baseline

- The customer-experience baseline used 95,830 orders with both valid review scores and delivery-status information.
- On-time orders had an average review score of approximately 4.29, compared with 2.57 for late orders.
- The difference in average review score was approximately 1.72 points on the 1–5 review scale.
- A Mann-Whitney U test found a statistically significant difference in review-score distributions between on-time and late orders.
- The reported p-value displayed as 0.0 because it was smaller than the numerical precision shown by Python, not because the true probability is literally zero.
- Delivery delay and review score had a Spearman correlation of -0.176.
- The negative correlation indicates that later delivery relative to the estimated date tends to be associated with lower review scores, although the relationship is relatively weak when delay is considered continuously.
- Delivery performance remains the strongest customer-experience relationship identified in the initial EDA and baseline analysis.
- These results represent associations rather than causal effects.

## Seller-Performance Baseline

- The seller-performance baseline included all 3,095 sellers.
- `seller_total_sales` was used as the initial seller-performance outcome.
- Seller order count had the strongest relationship with total sales, with a Spearman correlation of 0.853.
- Seller product count also had a strong positive relationship with total sales, with a correlation of 0.786.
- Seller category count had a moderate positive relationship with total sales, with a correlation of 0.506.
- Seller average item value had a moderate positive relationship with total sales, with a correlation of 0.490.
- All four relationships were statistically significant.
- The very small p-values for order count and product count were displayed as 0.0 because they were below Python's displayed numerical precision.
- The strong relationship between seller order count and total sales is expected because sellers with more orders have more opportunities to generate sales.
- These simple correlations provide a baseline rather than a complete explanation of seller performance.

## Next Analytical Step

- The baseline methods intentionally examine variables individually.
- Later analysis should determine whether the observed relationships remain meaningful after multiple operational, product, seller, and geographic factors are considered together.
- Improved methods should be compared with these baseline results to determine what additional information they provide and what limitations remain.